# Train the ECG models on a free Colab GPU

1. Upload `ecg-ai-fusion.zip` to the top level of your Google Drive (**My Drive**).
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run the cells from top to bottom. Each training cell saves its checkpoints to Drive, so a disconnect only loses the cell that was running.
4. At the end, download `ecg_trained_outputs.zip` from Drive and unzip it inside the project folder on your Mac (it contains `artifacts/`, `results/` and a test-fold copy of the data in `data/cache/` for the website's demo picker).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU: change the runtime type to T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/ecg-ai-fusion.zip -d /content/
%cd /content/ecg-ai-fusion
!mkdir -p /content/drive/MyDrive/ecg_outputs

In [ ]:
!pip install -q -r requirements.txt

## 1. Check the code (1–2 minutes, random tensors, no dataset)

In [ ]:
!python scripts/smoke_test.py

## 2. Download PTB-XL v1.0.3 (100 Hz records + metadata only)
Uses PhysioNet's open-data mirror on AWS S3. Re-running the cell resumes an interrupted download.

In [ ]:
!pip install -q awscli
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/ptb-xl/1.0.3/ data/ptb-xl --exclude "records500/*"
!ls data/ptb-xl && du -sh data/ptb-xl

If the S3 command fails, use the PhysioNet zip instead (larger, includes 500 Hz files):
```
!wget -q -O ptbxl.zip https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
!unzip -q ptbxl.zip -d data/ && mv data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 data/ptb-xl
```

## 3. Prepare data (labels, official folds, training-only normalization)

In [ ]:
!python scripts/prepare_data.py

## 4. Timing test (1 epoch per model)

In [ ]:
!python scripts/train_all.py --time-test

## 5. Train — one cell per seed, each saved to Drive

In [ ]:
!python scripts/train_all.py --seeds 0
!rsync -a artifacts results /content/drive/MyDrive/ecg_outputs/

In [ ]:
!python scripts/train_all.py --seeds 1
!rsync -a artifacts results /content/drive/MyDrive/ecg_outputs/

In [ ]:
!python scripts/train_all.py --seeds 2
!rsync -a artifacts results /content/drive/MyDrive/ecg_outputs/

If Colab disconnected, re-run section 0 and 3, then restore earlier checkpoints with:
```
!rsync -a /content/drive/MyDrive/ecg_outputs/artifacts /content/drive/MyDrive/ecg_outputs/results ./
```
and continue with the next seed.

## 6. Evaluate, explain, test

In [ ]:
!python scripts/evaluate_all.py
!python scripts/run_xai.py
!python -m pytest -q

## 7. Package the trained models and results for your Mac

In [ ]:
!zip -r -q /content/drive/MyDrive/ecg_trained_outputs.zip artifacts results data/cache/test_X.npy data/cache/test_meta.csv -x "artifacts/models/smoke/*" "artifacts/models/timing/*" "results/preds/*"
!ls -lh /content/drive/MyDrive/ecg_trained_outputs.zip